In [1]:
from openai import OpenAI
import pandas as pd
import os
from tqdm import tqdm

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="local_gpu"
)

MAX_TOKENS = 60
TEMPERATURE = 0.1
BATCH_SIZE = 500

In [2]:
# 精炼Prompt：给原有长文本，让模型压缩成短核心
def refine_prompt(title, long_text):
    prompt = f"""Title: {title}
Original text: {long_text}
Rewrite into ONE short sentence under 40 English words, only keep core attributes: genre, theme, target audience.
No extra explanation, only the short sentence.
"""
    return prompt

if __name__ == "__main__":
    INPUT_CSV = "./movie_book_cdsr_processed/item_meta_llm_enhanced.csv"
    # 精炼后新文件
    OUTPUT_CSV = "./movie_book_cdsr_processed/item_meta_llm_refined.csv"

    df = pd.read_csv(INPUT_CSV)
    print(f"总数据：{len(df)}")

    # 断点续跑
    if os.path.exists(OUTPUT_CSV):
        print("加载已精炼数据，断点续跑")
        done = pd.read_csv(OUTPUT_CSV)
        df = pd.merge(df, done[["item_id","llm_refined"]], on="item_id", how="left")
    else:
        df["llm_refined"] = ""
        df["llm_refined"] = df["llm_refined"].astype("string")

    # 待精炼
    need_refine = df[df["llm_refined"].isna() | (df["llm_refined"]=="")]
    print(f"待精炼条数：{len(need_refine)}")

    if len(need_refine) == 0:
        print("全部精炼完成")
        exit()

    for idx, row in enumerate(tqdm(need_refine.itertuples(index=False), total=len(need_refine))):
        item_id = row.item_id
        title = row.title
        long_text = row.llm_enhanced_text

        try:
            resp = client.chat.completions.create(
                model="qwen:0.5b",
                messages=[
                    {"role":"system","content":"Condense text into short core sentence."},
                    {"role":"user","content":refine_prompt(title, long_text)}
                ],
                max_tokens=MAX_TOKENS,
                temperature=TEMPERATURE
            )
            short_text = resp.choices[0].message.content.strip()
            df.loc[df.item_id==item_id, "llm_refined"] = short_text
        except:
            df.loc[df.item_id==item_id, "llm_refined"] = ""

        if idx % BATCH_SIZE == 0:
            df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
    print("二次精炼完成")

总数据：95661
加载已精炼数据，断点续跑
待精炼条数：35161


100%|██████████| 35161/35161 [3:29:01<00:00,  2.80it/s]  


二次精炼完成
